# Day 2 — Data Cleaning + SQLite Loading

This notebook validates the outputs produced by `scripts/day2_build_sqlite.py`.

Includes: missing values, duplicates, summary stats, category & fund house counts, NAV trend plots (sample), and SQLite sanity checks.

In [ ]:
from pathlib import Pathimport pandas as pdimport matplotlib.pyplot as pltimport sqlite3PROJECT_ROOT = Path('..').resolve()PROCESSED_NAV = PROJECT_ROOT / 'data' / 'processed' / 'nav_history_combined.csv'DB_PATH = PROJECT_ROOT / 'data' / 'db' / 'bluestock_mf.db'print('nav file exists:', PROCESSED_NAV.exists())print('db exists:', DB_PATH.exists())

In [ ]:
nav = pd.read_csv(PROCESSED_NAV)nav.head()nav.dtypes

In [ ]:
missing = nav.isna().sum().sort_values(ascending=False)missing[missing > 0].head(30)dup_count = nav.duplicated().sum()dup_count

In [ ]:
if 'nav' in nav.columns:    print('nav stats:')    display(nav['nav'].describe())    print('negative nav rows:', int((nav['nav'] < 0).sum()))if 'date' in nav.columns:    parsed = pd.to_datetime(nav['date'], errors='coerce')    print('date parse failures:', int(parsed.isna().sum()))

In [ ]:
if 'scheme_category' in nav.columns:    cat_counts = nav['scheme_category'].value_counts().head(30)    cat_countsif 'fund_house' in nav.columns:    house_counts = nav['fund_house'].value_counts().head(30)    house_counts

In [ ]:
if {'scheme_code', 'date', 'nav'}.issubset(nav.columns):    tmp = nav.copy()    tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce')    tmp = tmp.dropna(subset=['date', 'nav'])    sample_codes = tmp['scheme_code'].value_counts().head(3).index.tolist()    plt.figure(figsize=(12, 6))    for code in sample_codes:        s = tmp[tmp['scheme_code'] == code].sort_values('date')        plt.plot(s['date'], s['nav'], label=str(code), linewidth=1)    plt.title('NAV trend (sample schemes)')    plt.xlabel('Date')    plt.ylabel('NAV')    plt.legend(title='scheme_code')    plt.tight_layout()    plt.show()

In [ ]:
# SQLite sanity checkscon = sqlite3.connect(DB_PATH)cur = con.cursor()for tbl in ['dim_fund', 'dim_category', 'fact_nav', 'fact_transactions', 'fact_performance']:    try:        cur.execute(f'SELECT COUNT(*) FROM {tbl}')        print(tbl, cur.fetchone()[0])    except Exception as e:        print(tbl, 'error:', e)# Show fact_nav schemacur.execute('SELECT name, type, notnull, dflt_value, pk FROM pragma_table_info("fact_nav")')print('fact_nav columns:')for row in cur.fetchall():    print(row)con.close()